# Dataset Split Creation

This script can be used to create .json files as required by the ResNet-50_Convnext-Tiny_Classification.py script to facilitate training / validation / testing.

**NOTE:** This script might need to be extended to cater for other ways in which data is stored so that the .json can be built

In [19]:
import argparse
import json
import random
from pathlib import Path

# ============================================================
# Helpers
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def list_images(directory):
    return sorted([
        p for p in Path(directory).rglob("*")
        if p.suffix.lower() in IMG_EXTS
    ])

def list_images_from_dirs(directories):
    images = []
    for d in directories:
        images.extend(list_images(d))
    return images

def sample_images(images, n, rng):
    if len(images) < n:
        raise ValueError(
            f"Requested {n} images, but only {len(images)} available."
        )
    return rng.sample(images, n)

# ============================================================
# Main
# ============================================================

def create_multiclass_json_multi_dir(seed, label_config, out_json):
    """
    Creates train/val/test JSON splits for an arbitrary number of labels.

    Each label may be associated with multiple directories. Sampling is
    performed independently per directory, with fixed numbers of images
    assigned to train/val/test for each directory.

    Parameters
    ----------
    seed : int
        Random seed for reproducibility.
    label_config : dict
        Mapping from integer label -> configuration dict with keys:
            - "name": str (optional, for logging)
            - "dirs": list of directory paths
            - "train": int
            - "val": int
            - "test": int
    out_json : str or Path
        Output path for the generated JSON file.
    """
    rng = random.Random(seed)

    data = {
        "train": [],
        "val": [],
        "test": [],
    }

    for label, cfg in label_config.items():
        label_name = cfg.get("name", f"label_{label}")
        dirs = cfg["dirs"]
        n_train = cfg["train"]
        n_val = cfg["val"]
        n_test = cfg["test"]

        print(f"\nProcessing label {label} ({label_name})")

        for d in dirs:
            imgs = list_images(d)
            total = n_train + n_val + n_test

            if len(imgs) < total:
                raise ValueError(
                    f"Directory {d} has {len(imgs)} images, "
                    f"but {total} are required."
                )

            selected = rng.sample(imgs, total)

            train = selected[:n_train]
            val = selected[n_train:n_train + n_val]
            test = selected[n_train + n_val:]

            data["train"].extend(
                {"image": str(p.resolve()), "label": label}
                for p in train
            )
            data["val"].extend(
                {"image": str(p.resolve()), "label": label}
                for p in val
            )
            data["test"].extend(
                {"image": str(p.resolve()), "label": label}
                for p in test
            )

            print(
                f"  {Path(d).name}: "
                f"train={len(train)}, val={len(val)}, test={len(test)}"
            )

    # Shuffle within each split
    for split in data.values():
        rng.shuffle(split)

    # Save JSON
    out_path = Path(out_json)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"\nSaved split JSON to: {out_path}")
    print("Final split sizes:")
    for split_name, split_data in data.items():
        counts = {}
        for x in split_data:
            counts[x["label"]] = counts.get(x["label"], 0) + 1
        print(f"  {split_name}: {counts}")


### Example Code - Gender

In [20]:
train_amount = 289 
val_amount = 96
test_amount = 385
seed = 42
output_path = r'test_splits_gender_multiple_dirs.json'
label_config = {
    0: {
        "name": "Male",
        "dirs": [
            r"G:\Thesis\ImageRetrieval\ProfessionsCleaned\Male_Doctor",
            r"G:\Thesis\ImageRetrieval\ProfessionsCleaned\Male_Chef",
        ],
        "train": train_amount,
        "val": val_amount,
        "test": test_amount,
    },
    1: {
        "name": "Female",
        "dirs": [
            r"G:\Thesis\ImageRetrieval\ProfessionsCleaned\Female_Doctor",
            r"G:\Thesis\ImageRetrieval\ProfessionsCleaned\Female_Chef",
        ],
        "train": train_amount,
        "val": val_amount,
        "test": test_amount,
    }
}

create_multiclass_json_multi_dir(
    seed=seed,
    label_config=label_config,
    out_json=output_path,
)


Processing label 0 (Male)
  Male_Doctor: train=289, val=96, test=385
  Male_Chef: train=289, val=96, test=385

Processing label 1 (Female)
  Female_Doctor: train=289, val=96, test=385
  Female_Chef: train=289, val=96, test=385

Saved split JSON to: test_splits_gender_multiple_dirs.json
Final split sizes:
  train: {1: 578, 0: 578}
  val: {1: 192, 0: 192}
  test: {1: 770, 0: 770}


### Example Code - Skin Tone

In [ ]:
train_amount = 289 
val_amount = 96
test_amount = 385
seed = 42
output_path = r'test_splits_skin_tone_multiple_dirs.json'
label_config = {
    0: {
        "name": "Skin Tone Group 1",
        "dirs": [
            r"SkinToneDir1",
            r"SkinToneDir2",
        ],
        "train": train_amount,
        "val": val_amount,
        "test": test_amount,
    },
    1: {
        "name": "Skin Tone Group 2",
        "dirs": [
            r"SkinToneDir1",
            r"SkinToneDir2",
        ],
        "train": train_amount,
        "val": val_amount,
        "test": test_amount,
    },
    2: {
        "name": "Skin Tone Group 3",
        "dirs": [
            r"SkinToneDir1",
            r"SkinToneDir2",
        ],
        "train": train_amount,
        "val": val_amount,
        "test": test_amount,
    }
}

create_multiclass_json_multi_dir(
    seed=seed,
    label_config=label_config,
    out_json=output_path,
)